# 10 — Build future data and smoke-test inference

This notebook **demonstrates** reusable Python modules. Keep `PERSIST=False` for the first run. No training, model registration, or prediction-table writing occurs here.

## 1. Setup
Run on Databricks compute with project dependencies installed. This notebook uses the Git-folder code on the current branch.

In [0]:
import os
import site
from pathlib import Path

username = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)

project_root = Path(
    f"/Workspace/Users/{username}/finance-timeseries-ml-platform"
)

os.chdir(project_root)

site.addsitedir("src")

# print("Project root:", project_root)

In [0]:
%uv sync --frozen

In [0]:
%restart_python

In [0]:
from pathlib import Path
import sys
import mlflow
from pyspark.sql import functions as F

username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
project_root = Path(f"/Workspace/Users/{username}/finance-timeseries-ml-platform")
assert (project_root / "src" / "finance_ml").is_dir(), project_root
sys.path.insert(0, str(project_root / "src"))

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

from finance_ml.pipelines.future_data_pipeline import prepare_future_data
from finance_ml.pipelines.inference_pipeline import INFERENCE_CONFIG, run_inference_pipeline
# print("Using project:", project_root)

## 2. Configuration
The dry run returns Spark DataFrames without creating or overwriting Unity Catalog tables.

In [0]:
N_FUTURE_DAYS = 14
SEED = 84
PERSIST = False  # Keep False until data validation and an explicit persistence decision.
RUN_INFERENCE_SMOKE_TEST = True

## 3. Generate future Bronze, Silver, Gold, and sequence context

In [0]:
future_result = prepare_future_data(
    spark,
    history_silver_table="finance_ml.silver.account_events_clean",
    n_future_days=N_FUTURE_DAYS,
    seed=SEED,
    persist=PERSIST,
)
print("History end:", future_result["history_end"])
print("Future start:", future_result["future_start"])

## 4. Validate before persistence or inference

In [0]:
history_end = future_result["history_end"]
future_start = future_result["future_start"]
assert future_start.date() > history_end.date()

for layer in ("bronze", "silver", "gold", "context"):
    frame = future_result[layer]
    summary = frame.agg(
        F.count("*").alias("rows"),
        F.countDistinct("account_id").alias("accounts"),
        F.min("date").alias("first_date"),
        F.max("date").alias("last_date"),
    ).first()
    duplicates = (frame.groupBy("account_id", "date")
                  .count().filter(F.col("count") > 1).count())
    assert duplicates == 0, f"Duplicate account-dates in {layer}"
    print(f"{layer}: {summary.asDict()}")

for layer in ("bronze", "silver", "gold"):
    frame = future_result[layer]
    assert "risk_state" not in frame.columns
    assert "risk_state_name" not in frame.columns
    assert frame.filter(F.col("date") <= F.lit(history_end)).limit(1).count() == 0

n_accounts = future_result["bronze"].select("account_id").distinct().count()
assert future_result["bronze"].count() == n_accounts * N_FUTURE_DAYS
assert future_result["silver"].count() == n_accounts * N_FUTURE_DAYS
assert future_result["gold"].count() == n_accounts * N_FUTURE_DAYS
print("Basic read-only validation passed")

## 5. Preview
The context includes six historical Gold rows per account in addition to the 14 future dates. These historical observations must not be treated as new predictions.

In [0]:
display(future_result["gold"].orderBy("account_id", "date").limit(28))

## 6. Registered-model inference smoke test
The actual inference is performed in `finance_ml.pipelines.inference_pipeline.run_inference_pipeline`. The supplied in-memory DataFrame avoids writing an intermediate table. This can be memory-intensive on the driver; use the existing 2,000-account toy dataset only. Predictions are filtered to the future period.

In [0]:
if RUN_INFERENCE_SMOKE_TEST:
    result = run_inference_pipeline(
        config={**INFERENCE_CONFIG, "model_version": "1", "query_method": "spark"},
        spark=spark,
        gold_df=future_result["context"],
    )
    future_predictions = [
        item for item in result["predictions"]
        if item["time"].date() > history_end.date()
    ]
    assert future_predictions
    assert len(future_predictions) <= n_accounts * N_FUTURE_DAYS
    print("Model:", result["model_name"], "version:", result["model_version"])
    print("Sequences:", result["n_sequences"])
    print("Predictions (all context):", result["n_predictions"])
    print("Predictions (future only):", len(future_predictions))
    print("Future range:", min(x["time"] for x in future_predictions), "→", max(x["time"] for x in future_predictions))
    for p in future_predictions[:5]:
        print(p["group"], p["time"], p["predicted_class"], p["n_contributions"])

## 7. Persistence is deliberately separate
`PERSIST=False` above is the safe default. To persist future data later, perform a separate reviewed run with `PERSIST=True`. The current pipeline writes future-specific tables using `overwrite`; verify their names and destination policy before enabling this. Predictions are **not persisted** by this notebook.

**Known limitation:** Boundary trimming may omit the last two dates of each account. We should address online/edge-window coverage before production-style daily scoring or Power BI publication.

The original training tables and UC model version 1 remain unchanged.

In [0]:
from finance_ml.pipelines.inference_pipeline import run_inference_pipeline

# Retaining predictions at all sequence positions.
full_result = run_inference_pipeline(
    config={"boundary_width": 0},
    spark=spark,
    gold_df=future_result["context"],
)

# Keeping only the newly generated future dates.
future_predictions_full = [
    p for p in full_result["predictions"]
    if p["time"] > future_result["history_end"]
]

dates = [p["time"] for p in future_predictions_full]

print("Sequences:", full_result["n_sequences"])
print("All context predictions:", full_result["n_predictions"])
print("Future-only predictions:", len(future_predictions_full))
print("First predicted date:", min(dates))
print("Last predicted date:", max(dates))

assert len(future_predictions_full) == 28_000
assert max(dates).date().isoformat() == "2026-03-15"

print("Full future-period coverage verified.")